# Practice: how image generators work
*AI from scratch, elective* · lesson: https://ihor.sh/ai/diffusion.html

A real diffusion model, small enough for a CPU: on scikit-learn's 1,797 handwritten digits (8 × 8 pixels), build
the noise schedule, noise images in one shot, write the training loss and the sampling step, check whether the
samples copy the training set, then add classifier-free guidance and measure what the guidance scale trades.
About half a minute. Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it; `hint(n)` and
`answer(n)` are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['abar = torch.cumprod(alpha, 0)'], 2: ['a = abar[t][:, None]', 'return a.sqrt() * x0 + (1 - a).sqrt() * noise'], 3: ['xt = q_sample(x0, t, noise)', 'return F.mse_loss(net(xt, t, y), noise)'], 4: ['x = (x - beta[t] / (1 - abar[t]).sqrt() * eps) / alpha[t].sqrt()', 'if t > 0: x = x + beta[t].sqrt() * torch.randn_like(x)'], 5: ['typical = torch.cdist(X[:100], X[100:200]).mean().item()'], 6: ['rates, spread = {}, {} then for w in [0, 1, 3]: s = sample(3, w=w, n=100)', 'the classifier wants pixels 0…1: rates[w] = (clf.predict(((s + 1) / 2).numpy()) == 3).mean(); spread[w] = s.std(dim=0).mean().item()'], 7: ['std = torch.exp(0.5 * logvar)', 'return mu + torch.randn_like(std) * std'], 8: ['kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)', 'return kl.mean()']}
_ANSWERS = {1: 'YWJhciA9IHRvcmNoLmN1bXByb2QoYWxwaGEsIDAp', 2: 'YSA9IGFiYXJbdF1bOiwgTm9uZV0KcmV0dXJuIGEuc3FydCgpICogeDAgKyAoMSAtIGEpLnNxcnQoKSAqIG5vaXNl', 3: 'eHQgPSBxX3NhbXBsZSh4MCwgdCwgbm9pc2UpCnJldHVybiBGLm1zZV9sb3NzKG5ldCh4dCwgdCwgeSksIG5vaXNlKQ==', 4: 'eCA9ICh4IC0gYmV0YVt0XSAvICgxIC0gYWJhclt0XSkuc3FydCgpICogZXBzKSAvIGFscGhhW3RdLnNxcnQoKQppZiB0ID4gMDoKICAgIHggPSB4ICsgYmV0YVt0XS5zcXJ0KCkgKiB0b3JjaC5yYW5kbl9saWtlKHgp', 5: 'dHlwaWNhbCA9IHRvcmNoLmNkaXN0KFhbOjEwMF0sIFhbMTAwOjIwMF0pLm1lYW4oKS5pdGVtKCk=', 6: 'cmF0ZXMsIHNwcmVhZCA9IHt9LCB7fQpmb3IgdyBpbiBbMCwgMSwgM106CiAgICBzID0gc2FtcGxlKDMsIHc9dywgbj0xMDApCiAgICByYXRlc1t3XSA9IChjbGYucHJlZGljdCgoKHMgKyAxKSAvIDIpLm51bXB5KCkpID09IDMpLm1lYW4oKQogICAgc3ByZWFkW3ddID0gcy5zdGQoZGltPTApLm1lYW4oKS5pdGVtKCk=', 7: 'c3RkID0gdG9yY2guZXhwKDAuNSAqIGxvZ3ZhcikKcmV0dXJuIG11ICsgdG9yY2gucmFuZG5fbGlrZShzdGQpICogc3Rk', 8: 'a2wgPSAtMC41ICogdG9yY2guc3VtKDEgKyBsb2d2YXIgLSBtdS5wb3coMikgLSBsb2d2YXIuZXhwKCksIGRpbT0xKQpyZXR1cm4ga2wubWVhbigp'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
digits = load_digits()
X = torch.tensor(digits.data / 8 - 1, dtype=torch.float32)     # 64 pixels per image, scaled to -1…1
Y = torch.tensor(digits.target)
clf = LogisticRegression(max_iter=3000).fit(digits.data / 16, digits.target)     # reads 8 × 8 digits, for checking samples
T = 200
beta = torch.linspace(1e-4, 0.05, T)                            # how much noise each step adds
alpha = 1 - beta
def show(images, title=""):
    fig, axes = plt.subplots(1, len(images), figsize=(len(images), 1.3))
    for ax, im in zip(axes, images):
        ax.imshow(np.asarray(im).reshape(8, 8), cmap="gray", vmin=-1, vmax=1); ax.axis("off")
    fig.suptitle(title); plt.show()
show(X[:10], "training digits")

## 1. The schedule
`abar[t]` is the product of `alpha[0]` … `alpha[t]`: how much image is left after `t + 1` steps. Set `abar` with
`torch.cumprod(alpha, 0)` (a running product, like `np.cumsum` is a running sum).

In [ ]:
abar = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print("image variance left after 1, 50, 100, 200 steps:", abar[[0, 49, 99, 199]] if abar is not None else None)

In [ ]:
check(1, lambda: abs(abar[0] - alpha[0]) < 1e-9 and abar[-1] < 0.01, "almost all image at the start, under 1% at the end")

## 2. Noise an image in one shot
Write `q_sample(x0, t, noise)`: the lesson's `sqrt(abar_t) · x0 + sqrt(1 − abar_t) · noise`, for a batch: `t` is a
tensor of steps, so `abar[t]` has one number per image; add `[:, None]` to line it up with the 64 pixels.

In [ ]:
def q_sample(x0, t, noise):
    # YOUR CODE HERE (stuck? hint(2), then answer(2))
    ...

if abar is not None:
    torch.manual_seed(0)
    show([q_sample(X[:1], torch.tensor([t]), torch.randn(1, 64))[0] for t in [0, 20, 50, 100, 150, 199]], "one digit at steps 0, 20, 50, 100, 150, 199")

In [ ]:
check(2, lambda: torch.allclose(q_sample(X[:3], torch.tensor([0, 0, 0]), torch.zeros(3, 64)), abar[0].sqrt() * X[:3])
      and abs(q_sample(X, torch.full((len(X),), T - 1), torch.randn(len(X), 64)).std().item() - 1) < 0.1,
      "step 0 is (almost) the image; the last step is noise with spread about 1")

## 3. The training loss
The network (given) sees a noisy image, its step (as sine waves) and a label (10 means "no label": dropped 10% of
the time, for guidance in section 6). Write the loss in `train_step`: noise the batch with `q_sample` at random steps
`t`, and return the mean squared error between the true noise and the network's guess `net(xt, t, y)`.

In [ ]:
class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.label = nn.Embedding(11, 32)
        self.f = nn.Sequential(nn.Linear(64 + 32 + 32, 256), nn.SiLU(), nn.Linear(256, 256), nn.SiLU(), nn.Linear(256, 64))
    def forward(self, x, t, y):
        freqs = torch.exp(-math.log(1000) * torch.arange(16) / 16)
        angles = t[:, None].float() * freqs[None]
        return self.f(torch.cat([x, torch.sin(angles), torch.cos(angles), self.label(y)], 1))

def train_step(net, x0, y):
    t = torch.randint(T, (len(x0),))
    noise = torch.randn_like(x0)
    # YOUR CODE HERE (stuck? hint(3), then answer(3))
    ...

torch.manual_seed(0); net = Denoiser(); opt = torch.optim.Adam(net.parameters(), lr=2e-3); losses = []
for step in range(6000):
    i = torch.randint(len(X), (128,)); y = Y[i].clone(); y[torch.rand(128) < 0.1] = 10
    loss = train_step(net, X[i], y)
    if loss is None:
        break
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
print("loss: first 100 steps", np.mean(losses[:100]) if losses else None, "· last 100", np.mean(losses[-100:]) if losses else None)

In [ ]:
check(3, lambda: np.mean(losses[-100:]) < 0.3, "from about 1 (guessing nothing) to under 0.3: it learned to see noise")

## 4. Sampling: one step back
Write the update inside `sample`, the DDPM step: from the predicted noise `eps`,
`x = (x − beta[t] / sqrt(1 − abar[t]) · eps) / sqrt(alpha[t])`, then (except at the last step) add fresh noise
`sqrt(beta[t]) · randn`. The guidance mix is given: with `w = 1` it's the plain conditional prediction.

In [ ]:
@torch.no_grad()
def sample(label, w=1.0, n=10, seed=0):
    torch.manual_seed(seed); x = torch.randn(n, 64)
    for t in reversed(range(T)):
        tt = torch.full((n,), t)
        cond, uncond = net(x, tt, torch.full((n,), label)), net(x, tt, torch.full((n,), 10))
        eps = uncond + w * (cond - uncond)                       # classifier-free guidance (section 6)
        # YOUR CODE HERE (stuck? hint(4), then answer(4))
        ...
    return x.clamp(-1, 1)

for label in [0, 3, 7]:
    show(sample(label), f"generated: {label}")

In [ ]:
s3 = sample(3, n=100)
check(4, lambda: (clf.predict(((s3 + 1) / 2).numpy()) == 3).mean() > 0.7, "a classifier reads them as 3s: digits drawn from nothing but noise and a label")

## 5. New digits, or copies?
For each of 100 generated 3s, the distance to its nearest training image (given). Set `typical`: the average distance
between two *different* training images, from the first 100 against the next 100 (`torch.cdist(A, B)` gives every pair).

In [ ]:
nearest = torch.cdist(s3, X).min(dim=1).values
typical = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
print("generated → nearest training image:", nearest.mean().item(), "· two training images apart:", typical, "· closest copy:", nearest.min().item())

In [ ]:
check(5, lambda: 0 < nearest.min().item() and nearest.mean().item() < typical,
      "closer to the training set than two random digits are to each other, but not copies: a network too small to memorize every pixel generalizes")

## 6. The guidance scale
The classifier from the setup reads the generated images. Set `rates`: for `w` in `[0, 1, 3]`, the share of 100 samples of
label 3 that the classifier calls a 3, and `spread`: the average per-pixel standard deviation across those 100
samples (variety: `samples.std(dim=0).mean().item()`).

In [ ]:
rates = spread = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
print("called a 3:", rates, "\nvariety:", spread)

In [ ]:
check(6, lambda: rates[0] < 0.5 < rates[1] <= rates[3] and spread[3] < spread[0],
      "w = 0 ignores the label (a few 3s by chance); w = 1 follows it; w = 3 follows it always, with less variety: the guidance trade-off")

## 7. Variational autoencoder: the reparameterization trick
Stable Diffusion compresses high-resolution images with a VAE into a latent space where diffusion runs.
The encoder outputs mean `mu` and `logvar` ($\log \sigma^2$). Write `reparameterize(mu, logvar)`: sample
$z = \mu + \sigma \odot \varepsilon$ where $\sigma = \exp(0.5 \cdot \text{logvar})$ and $\varepsilon \sim \mathcal{N}(0, I)$.
Moving the random draw to an external noise $\varepsilon$ makes $z$ differentiable with respect to `mu` and `logvar`.

In [ ]:
def reparameterize(mu, logvar):
    # YOUR CODE HERE (stuck? hint(7), then answer(7))
    ...

torch.manual_seed(42)
test_mu = torch.randn(10, 4, requires_grad=True)
test_logvar = torch.randn(10, 4, requires_grad=True)
test_z = reparameterize(test_mu, test_logvar)
if test_z is not None and test_z is not ...:
    test_z.sum().backward()
    print("reparameterized sample shape:", test_z.shape, "· mu grad:", test_mu.grad.norm().item())

In [ ]:
check(7, lambda: (
    abs(reparameterize(torch.zeros(2000, 2), torch.zeros(2000, 2)).std().item() - 1.0) < 0.08
    and test_mu.grad is not None
), "randomness moved to external normal draw; gradients backpropagate directly through mu")

## 8. VAE loss: Kullback-Leibler divergence to standard normal
To keep the latent space dense with no unmapped holes, the VAE penalizes divergence from $\mathcal{N}(0, I)$.
Write `vae_kl_loss(mu, logvar)`: the analytical KL divergence averaged over the batch:
$$D_{\text{KL}} = -\frac{1}{2 B} \sum_{i=1}^B \sum_{j=1}^D (1 + \text{logvar}_{i,j} - \mu_{i,j}^2 - \exp(\text{logvar}_{i,j}))$$

In [ ]:
def vae_kl_loss(mu, logvar):
    # YOUR CODE HERE (stuck? hint(8), then answer(8))
    ...

test_kl = vae_kl_loss(torch.zeros(5, 4), torch.zeros(5, 4))
print("KL at N(0, I):", test_kl.item() if test_kl is not None and test_kl is not ... else None)

In [ ]:
check(8, lambda: (
    torch.allclose(vae_kl_loss(torch.zeros(5, 4), torch.zeros(5, 4)), torch.tensor(0.0))
    and vae_kl_loss(torch.ones(5, 4), torch.zeros(5, 4)) > 1.5
), "zero KL penalty when encoder matches the standard normal prior; positive penalty when it drifts")

## Next
Back to the path: https://ihor.sh/ai/ · Hugging Face's "Annotated Diffusion Model": https://huggingface.co/blog/annotated-diffusion